In [40]:
import pandas as pd
import numpy as np
import requests
import io
import zipfile
import os

In [41]:
# To create a comprehensive list of all NDC-level information from the FDA:

# URL of the FDA NDC package
url = "https://www.accessdata.fda.gov/cder/ndcxls.zip"

# Fetch content and unzip in memory
response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # List files to find exact names (usually 'product.xls' and 'package.xls')
    print(z.namelist())
    
    # Load 'product.xls' into pandas, using sep='\t' for tab-delimited files
    with z.open('product.xls') as f:
        df_product = pd.read_csv(f, sep='\t', encoding='latin1')
        
     # Load 'package.xls' into pandas, using sep='\t' for tab-delimited files
    with z.open('package.xls') as f:
        df_package = pd.read_csv(f, sep='\t', encoding='latin1')


['package.xls', 'product.xls']


In [42]:
df_package.head()

,PRODUCTID,PRODUCTNDC,NDCPACKAGECODE,PACKAGEDESCRIPTION,STARTMARKETINGDATE,ENDMARKETINGDATE,NDC_EXCLUDE_FLAG,SAMPLE_PACKAGE
0,0002-0152_f4a0acea-cd2f-4e90-b495-bb07116e0509,0002-0152,0002-0152-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0002-0152-01)...",20240328,NaN,N,N
1,0002-0152_f4a0acea-cd2f-4e90-b495-bb07116e0509,0002-0152,0002-0152-04,"4 VIAL, SINGLE-DOSE in 1 CARTON (0002-0152-04)...",20240808,NaN,N,N
2,0002-0152_f4a0acea-cd2f-4e90-b495-bb07116e0509,0002-0152,0002-0152-61,"4 VIAL, SINGLE-DOSE in 1 CARTON (0002-0152-61)...",20250501,NaN,N,Y
3,0002-0213_42527ae4-c593-4e13-8b77-c0511198c708,0002-0213,0002-0213-01,"1 VIAL, MULTI-DOSE in 1 CARTON (0002-0213-01) ...",20230620,20261215.0,N,N
4,0002-0243_f4a0acea-cd2f-4e90-b495-bb07116e0509,0002-0243,0002-0243-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0002-0243-01)...",20240328,NaN,N,N


In [43]:
#Need to create the NDC11 and NDC9column 

df_package['NDC11'] = np.nan 

In [44]:
#modify NDCPACKAGECODE to NDC11 format and column

def convert_ndc_10_to_11(NDCPACKAGECODE):
    # Split by hyphen to identify the segment lengths
    segments = str(NDCPACKAGECODE).split('-')
    
    if len(segments) != 3:
        return "Invalid NDC format"

    # Apply padding rules based on segment lengths
    if len(segments[0]) == 4: # 4-4-2 format
        segments[0] = segments[0].zfill(5)
    elif len(segments[1]) == 3: # 5-3-2 format
        segments[1] = segments[1].zfill(4)
    elif len(segments[2]) == 1: # 5-4-1 format
        segments[2] = segments[2].zfill(2)
        
    # Join and remove hyphens for standard 11-digit format
    return "".join(segments)


# Apply function to create new column
df_package['NDC11'] = df_package['NDCPACKAGECODE'].apply(convert_ndc_10_to_11)

In [45]:
#create the product column by removing all non-numerics (which includes "-")

df_package['NDCProduct'] = df_package['PRODUCTNDC'].str.replace(r'\D', '', regex=True)

In [46]:
#rearrance the order of columns in the package dataframe:

df_package = df_package[[
    'PRODUCTID',  
    'PRODUCTNDC',   
    'NDCPACKAGECODE', 
    'NDC11',
    'NDCProduct',
    'PACKAGEDESCRIPTION',
    'STARTMARKETINGDATE',
    'ENDMARKETINGDATE',
    'NDC_EXCLUDE_FLAG',
    'SAMPLE_PACKAGE']]

In [47]:
df_package.head()

,PRODUCTID,PRODUCTNDC,NDCPACKAGECODE,NDC11,NDCProduct,PACKAGEDESCRIPTION,STARTMARKETINGDATE,ENDMARKETINGDATE,NDC_EXCLUDE_FLAG,SAMPLE_PACKAGE
0,0002-0152_f4a0acea-cd2f-4e90-b495-bb07116e0509,0002-0152,0002-0152-01,00002015201,00020152,"1 VIAL, SINGLE-DOSE in 1 CARTON (0002-0152-01)...",20240328,NaN,N,N
1,0002-0152_f4a0acea-cd2f-4e90-b495-bb07116e0509,0002-0152,0002-0152-04,00002015204,00020152,"4 VIAL, SINGLE-DOSE in 1 CARTON (0002-0152-04)...",20240808,NaN,N,N
2,0002-0152_f4a0acea-cd2f-4e90-b495-bb07116e0509,0002-0152,0002-0152-61,00002015261,00020152,"4 VIAL, SINGLE-DOSE in 1 CARTON (0002-0152-61)...",20250501,NaN,N,Y
3,0002-0213_42527ae4-c593-4e13-8b77-c0511198c708,0002-0213,0002-0213-01,00002021301,00020213,"1 VIAL, MULTI-DOSE in 1 CARTON (0002-0213-01) ...",20230620,20261215.0,N,N
4,0002-0243_f4a0acea-cd2f-4e90-b495-bb07116e0509,0002-0243,0002-0243-01,00002024301,00020243,"1 VIAL, SINGLE-DOSE in 1 CARTON (0002-0243-01)...",20240328,NaN,N,N


In [48]:
df_package.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 213866 entries, 0 to 213865
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   PRODUCTID           213866 non-null  object 
 1   PRODUCTNDC          213866 non-null  object 
 2   NDCPACKAGECODE      213866 non-null  object 
 3   NDC11               213866 non-null  object 
 4   NDCProduct          213866 non-null  object 
 5   PACKAGEDESCRIPTION  213866 non-null  object 
 6   STARTMARKETINGDATE  213866 non-null  int64  
 7   ENDMARKETINGDATE    5369 non-null    float64
 8   NDC_EXCLUDE_FLAG    213866 non-null  object 
 9   SAMPLE_PACKAGE      213866 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 16.3+ MB


In [49]:
df_product.head()

,PRODUCTID,PRODUCTNDC,PRODUCTTYPENAME,PROPRIETARYNAME,PROPRIETARYNAMESUFFIX,NONPROPRIETARYNAME,DOSAGEFORMNAME,ROUTENAME,STARTMARKETINGDATE,ENDMARKETINGDATE,MARKETINGCATEGORYNAME,APPLICATIONNUMBER,LABELERNAME,SUBSTANCENAME,ACTIVE_NUMERATOR_STRENGTH,ACTIVE_INGRED_UNIT,PHARM_CLASSES,DEASCHEDULE,NDC_EXCLUDE_FLAG,LISTING_RECORD_CERTIFIED_THROUGH
0,0002-0152_f4a0acea-cd2f-4e90-b495-bb07116e0509,0002-0152,HUMAN PRESCRIPTION DRUG,Zepbound,NaN,tirzepatide,"INJECTION, SOLUTION",SUBCUTANEOUS,20240328,NaN,NDA,NDA217806,Eli Lilly and Company,TIRZEPATIDE,2.5,mg/.5mL,"G-Protein-linked Receptor Interactions [MoA], ...",NaN,N,20271231.0
1,0002-0213_42527ae4-c593-4e13-8b77-c0511198c708,0002-0213,HUMAN OTC DRUG,Humulin,R,Insulin human,"INJECTION, SOLUTION",PARENTERAL,19830627,20261215.0,BLA,BLA018780,Eli Lilly and Company,INSULIN HUMAN,100,[iU]/mL,"Insulin [CS], Insulin [EPC]",NaN,N,NaN
2,0002-0243_f4a0acea-cd2f-4e90-b495-bb07116e0509,0002-0243,HUMAN PRESCRIPTION DRUG,Zepbound,NaN,tirzepatide,"INJECTION, SOLUTION",SUBCUTANEOUS,20240328,NaN,NDA,NDA217806,Eli Lilly and Company,TIRZEPATIDE,5,mg/.5mL,"G-Protein-linked Receptor Interactions [MoA], ...",NaN,N,20271231.0
3,0002-0800_3d52c48f-89cd-4d3d-8db6-1789e76a1c79,0002-0800,HUMAN PRESCRIPTION DRUG,Sterile Diluent,NaN,diluent,"INJECTION, SOLUTION",SUBCUTANEOUS,19870710,NaN,BLA,BLA020563,Eli Lilly and Company,WATER,1,mL/mL,NaN,NaN,N,20261231.0
4,0002-1152_371ba02d-d590-4295-a315-b5efe4596614,0002-1152,HUMAN PRESCRIPTION DRUG,Mounjaro,NaN,tirzepatide,"INJECTION, SOLUTION",SUBCUTANEOUS,20230728,NaN,NDA,NDA215866,Eli Lilly and Company,TIRZEPATIDE,2.5,mg/.5mL,"G-Protein-linked Receptor Interactions [MoA], ...",NaN,N,20271231.0


In [50]:
df_product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113310 entries, 0 to 113309
Data columns (total 20 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   PRODUCTID                         113310 non-null  object 
 1   PRODUCTNDC                        113310 non-null  object 
 2   PRODUCTTYPENAME                   113310 non-null  object 
 3   PROPRIETARYNAME                   113302 non-null  object 
 4   PROPRIETARYNAMESUFFIX             8544 non-null    object 
 5   NONPROPRIETARYNAME                113307 non-null  object 
 6   DOSAGEFORMNAME                    113310 non-null  object 
 7   ROUTENAME                         111516 non-null  object 
 8   STARTMARKETINGDATE                113310 non-null  int64  
 9   ENDMARKETINGDATE                  3504 non-null    float64
 10  MARKETINGCATEGORYNAME             113310 non-null  object 
 11  APPLICATIONNUMBER                 97263 non-null   o

In [51]:
#the 'package' and 'product' files are both now loaded
#these files contain the baseline NDC-level information for the entire master drug file

In [52]:
#each file contains a line-specific identifier in the 'PRODUCTID' field which we can use to join

In [53]:
#left join on the ProductID associated with each file 
NDC = df_package.merge(df_product, on='PRODUCTID', how='left') 

In [54]:
#no longer need NDC-join-related information, so will drop the 'ProductID' field
NDC = NDC.drop(['PRODUCTID'], axis=1)

In [55]:
NDC.head()

,PRODUCTNDC_x,NDCPACKAGECODE,NDC11,NDCProduct,PACKAGEDESCRIPTION,STARTMARKETINGDATE_x,ENDMARKETINGDATE_x,NDC_EXCLUDE_FLAG_x,SAMPLE_PACKAGE,PRODUCTNDC_y,...,MARKETINGCATEGORYNAME,APPLICATIONNUMBER,LABELERNAME,SUBSTANCENAME,ACTIVE_NUMERATOR_STRENGTH,ACTIVE_INGRED_UNIT,PHARM_CLASSES,DEASCHEDULE,NDC_EXCLUDE_FLAG_y,LISTING_RECORD_CERTIFIED_THROUGH
0,0002-0152,0002-0152-01,00002015201,00020152,"1 VIAL, SINGLE-DOSE in 1 CARTON (0002-0152-01)...",20240328,NaN,N,N,0002-0152,...,NDA,NDA217806,Eli Lilly and Company,TIRZEPATIDE,2.5,mg/.5mL,"G-Protein-linked Receptor Interactions [MoA], ...",NaN,N,20271231.0
1,0002-0152,0002-0152-04,00002015204,00020152,"4 VIAL, SINGLE-DOSE in 1 CARTON (0002-0152-04)...",20240808,NaN,N,N,0002-0152,...,NDA,NDA217806,Eli Lilly and Company,TIRZEPATIDE,2.5,mg/.5mL,"G-Protein-linked Receptor Interactions [MoA], ...",NaN,N,20271231.0
2,0002-0152,0002-0152-61,00002015261,00020152,"4 VIAL, SINGLE-DOSE in 1 CARTON (0002-0152-61)...",20250501,NaN,N,Y,0002-0152,...,NDA,NDA217806,Eli Lilly and Company,TIRZEPATIDE,2.5,mg/.5mL,"G-Protein-linked Receptor Interactions [MoA], ...",NaN,N,20271231.0
3,0002-0213,0002-0213-01,00002021301,00020213,"1 VIAL, MULTI-DOSE in 1 CARTON (0002-0213-01) ...",20230620,20261215.0,N,N,0002-0213,...,BLA,BLA018780,Eli Lilly and Company,INSULIN HUMAN,100,[iU]/mL,"Insulin [CS], Insulin [EPC]",NaN,N,NaN
4,0002-0243,0002-0243-01,00002024301,00020243,"1 VIAL, SINGLE-DOSE in 1 CARTON (0002-0243-01)...",20240328,NaN,N,N,0002-0243,...,NDA,NDA217806,Eli Lilly and Company,TIRZEPATIDE,5,mg/.5mL,"G-Protein-linked Receptor Interactions [MoA], ...",NaN,N,20271231.0


In [56]:
NDC.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 213866 entries, 0 to 213865
Data columns (total 28 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   PRODUCTNDC_x                      213866 non-null  object 
 1   NDCPACKAGECODE                    213866 non-null  object 
 2   NDC11                             213866 non-null  object 
 3   NDCProduct                        213866 non-null  object 
 4   PACKAGEDESCRIPTION                213866 non-null  object 
 5   STARTMARKETINGDATE_x              213866 non-null  int64  
 6   ENDMARKETINGDATE_x                5369 non-null    float64
 7   NDC_EXCLUDE_FLAG_x                213866 non-null  object 
 8   SAMPLE_PACKAGE                    213866 non-null  object 
 9   PRODUCTNDC_y                      213864 non-null  object 
 10  PRODUCTTYPENAME                   213864 non-null  object 
 11  PROPRIETARYNAME                   213852 non-null  o

In [57]:
#remove duplicate columns after join

NDC = NDC.drop(columns=['PRODUCTNDC_y', 'STARTMARKETINGDATE_y', 'ENDMARKETINGDATE_y', 'NDC_EXCLUDE_FLAG_y'], axis = 1)

In [58]:
#rename columns that were duplicated during join

NDC.rename(columns={'PRODUCTNDC_x': 'PRODUCTNDC', 'STARTMARKETINGDATE_x': 'STARTMARKETINGDATE', 'ENDMARKETINGDATE_x': 'ENDMARKETINGDATE', 'NDC_EXCLUDE_FLAG_x': 'NDC_EXCLUDE_FLAG'}, inplace=True)

In [59]:
#start and end marketing dates are in int format, need to convert to date

# Convert using the specific format
NDC['STARTMARKETINGDATE'] = pd.to_datetime(NDC['STARTMARKETINGDATE'], format='%Y%m%d')

NDC['ENDMARKETINGDATE'] = pd.to_datetime(NDC['ENDMARKETINGDATE'], format='%Y%m%d')

In [60]:
NDC.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 213866 entries, 0 to 213865
Data columns (total 24 columns):
 #   Column                            Non-Null Count   Dtype         
---  ------                            --------------   -----         
 0   PRODUCTNDC                        213866 non-null  object        
 1   NDCPACKAGECODE                    213866 non-null  object        
 2   NDC11                             213866 non-null  object        
 3   NDCProduct                        213866 non-null  object        
 4   PACKAGEDESCRIPTION                213866 non-null  object        
 5   STARTMARKETINGDATE                213866 non-null  datetime64[ns]
 6   ENDMARKETINGDATE                  5369 non-null    datetime64[ns]
 7   NDC_EXCLUDE_FLAG                  213866 non-null  object        
 8   SAMPLE_PACKAGE                    213866 non-null  object        
 9   PRODUCTTYPENAME                   213864 non-null  object        
 10  PROPRIETARYNAME                 

In [61]:
#the drug-related NDC file is now created, 
#need to now begin to add-in additional oncology-related information

In [62]:
#Download the most recent oncology-NDC information from SEER:
#https://seer.cancer.gov/oncologytoolbox/canmed/ndconc/?ordering=None&&page=490

url2 = "https://seer.cancer.gov/oncologytoolbox/canmed/ndconc/?&_export"
SEER_NDC = pd.read_csv(url2)

In [63]:
SEER_NDC.head()

,NDC-11 (Package),NDC-9 (Product),Generic Name,Brand Name,Strength,SEER*Rx Category,Major Class,Minor Class,Administration Route,Package Effective Date,Package Discontinuation Date,Status
0,00002-1717-28,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,10/16/2025,NaN,In Use
1,00002-1717-56,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,10/15/2025,NaN,In Use
2,00002-1717-61,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,11/13/2025,NaN,In Use
3,00002-2980-26,00002-2980,Selpercatinib,RETEVMO,80.0 mg/1,Chemotherapy,Tyrosine Kinase Inhibitor,VEGFR,Oral,05/08/2020,NaN,In Use
4,00002-2980-60,00002-2980,Selpercatinib,RETEVMO,80.0 mg/1,Chemotherapy,Tyrosine Kinase Inhibitor,VEGFR,Oral,05/08/2020,NaN,In Use


In [64]:
SEER_NDC.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12250 entries, 0 to 12249
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   NDC-11 (Package)              12250 non-null  object
 1   NDC-9 (Product)               12250 non-null  object
 2   Generic Name                  12250 non-null  object
 3   Brand Name                    12250 non-null  object
 4   Strength                      11406 non-null  object
 5   SEER*Rx Category              12250 non-null  object
 6   Major Class                   12250 non-null  object
 7   Minor Class                   10703 non-null  object
 8   Administration Route          11971 non-null  object
 9   Package Effective Date        12241 non-null  object
 10  Package Discontinuation Date  3832 non-null   object
 11  Status                        12250 non-null  object
dtypes: object(12)
memory usage: 1.1+ MB


In [65]:
# Create an NDC11 column for the SEER_NDC file to use as a join feature

SEER_NDC['NDC11'] = np.nan 

In [66]:
#remove the non-numberical characters from NDC-11 to create the new NDC11 column
#create the product column by removing all non-numerics (which includes "-")

SEER_NDC['NDC11'] = SEER_NDC['NDC-11 (Package)'].str.replace(r'\D', '', regex=True)

In [67]:
#rearrance columns 

SEER_NDC = SEER_NDC[[
    'NDC11', 
    'NDC-11 (Package)',
    'NDC-9 (Product)',
    'Generic Name',
    'Brand Name',
    'Strength',
    'SEER*Rx Category',
    'Major Class',
    'Minor Class',
    'Administration Route',
    'Package Effective Date',
    'Package Discontinuation Date',
    'Status',
    ]]

In [68]:
SEER_NDC.head()

,NDC11,NDC-11 (Package),NDC-9 (Product),Generic Name,Brand Name,Strength,SEER*Rx Category,Major Class,Minor Class,Administration Route,Package Effective Date,Package Discontinuation Date,Status
0,00002171728,00002-1717-28,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,10/16/2025,NaN,In Use
1,00002171756,00002-1717-56,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,10/15/2025,NaN,In Use
2,00002171761,00002-1717-61,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,11/13/2025,NaN,In Use
3,00002298026,00002-2980-26,00002-2980,Selpercatinib,RETEVMO,80.0 mg/1,Chemotherapy,Tyrosine Kinase Inhibitor,VEGFR,Oral,05/08/2020,NaN,In Use
4,00002298060,00002-2980-60,00002-2980,Selpercatinib,RETEVMO,80.0 mg/1,Chemotherapy,Tyrosine Kinase Inhibitor,VEGFR,Oral,05/08/2020,NaN,In Use


In [69]:
#adjust date columns to datetime format

SEER_NDC['Package Effective Date'] = pd.to_datetime(SEER_NDC['Package Effective Date'])

SEER_NDC['Package Discontinuation Date'] = pd.to_datetime(SEER_NDC['Package Discontinuation Date'])

In [70]:
SEER_NDC.head()

,NDC11,NDC-11 (Package),NDC-9 (Product),Generic Name,Brand Name,Strength,SEER*Rx Category,Major Class,Minor Class,Administration Route,Package Effective Date,Package Discontinuation Date,Status
0,00002171728,00002-1717-28,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,2025-10-16,NaT,In Use
1,00002171756,00002-1717-56,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,2025-10-15,NaT,In Use
2,00002171761,00002-1717-61,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,2025-11-13,NaT,In Use
3,00002298026,00002-2980-26,00002-2980,Selpercatinib,RETEVMO,80.0 mg/1,Chemotherapy,Tyrosine Kinase Inhibitor,VEGFR,Oral,2020-05-08,NaT,In Use
4,00002298060,00002-2980-60,00002-2980,Selpercatinib,RETEVMO,80.0 mg/1,Chemotherapy,Tyrosine Kinase Inhibitor,VEGFR,Oral,2020-05-08,NaT,In Use


In [71]:
SEER_NDC.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12250 entries, 0 to 12249
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   NDC11                         12250 non-null  object        
 1   NDC-11 (Package)              12250 non-null  object        
 2   NDC-9 (Product)               12250 non-null  object        
 3   Generic Name                  12250 non-null  object        
 4   Brand Name                    12250 non-null  object        
 5   Strength                      11406 non-null  object        
 6   SEER*Rx Category              12250 non-null  object        
 7   Major Class                   12250 non-null  object        
 8   Minor Class                   10703 non-null  object        
 9   Administration Route          11971 non-null  object        
 10  Package Effective Date        12241 non-null  datetime64[ns]
 11  Package Discontinuation Date

In [104]:
#join the NDC file to SEER_NDC to obtain baseline oncology information for each NDC
SEER_NDC_File = pd.merge(SEER_NDC, NDC, on='NDC11', how='left')

In [105]:
SEER_NDC_File.head()

,NDC11,NDC-11 (Package),NDC-9 (Product),Generic Name,Brand Name,Strength,SEER*Rx Category,Major Class,Minor Class,Administration Route,...,ROUTENAME,MARKETINGCATEGORYNAME,APPLICATIONNUMBER,LABELERNAME,SUBSTANCENAME,ACTIVE_NUMERATOR_STRENGTH,ACTIVE_INGRED_UNIT,PHARM_CLASSES,DEASCHEDULE,LISTING_RECORD_CERTIFIED_THROUGH
0,00002171728,00002-1717-28,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,...,ORAL,NDA,NDA218881,Eli Lilly and Company,IMLUNESTRANT,200,mg/1,Breast Cancer Resistance Protein Inhibitors [M...,NaN,20271231.0
1,00002171756,00002-1717-56,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,...,ORAL,NDA,NDA218881,Eli Lilly and Company,IMLUNESTRANT,200,mg/1,Breast Cancer Resistance Protein Inhibitors [M...,NaN,20271231.0
2,00002171761,00002-1717-61,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,...,ORAL,NDA,NDA218881,Eli Lilly and Company,IMLUNESTRANT,200,mg/1,Breast Cancer Resistance Protein Inhibitors [M...,NaN,20271231.0
3,00002298026,00002-2980-26,00002-2980,Selpercatinib,RETEVMO,80.0 mg/1,Chemotherapy,Tyrosine Kinase Inhibitor,VEGFR,Oral,...,ORAL,NDA,NDA213246,Eli Lilly and Company,SELPERCATINIB,80,mg/1,Breast Cancer Resistance Protein Inhibitors [M...,NaN,20261231.0
4,00002298060,00002-2980-60,00002-2980,Selpercatinib,RETEVMO,80.0 mg/1,Chemotherapy,Tyrosine Kinase Inhibitor,VEGFR,Oral,...,ORAL,NDA,NDA213246,Eli Lilly and Company,SELPERCATINIB,80,mg/1,Breast Cancer Resistance Protein Inhibitors [M...,NaN,20261231.0


In [106]:
SEER_NDC_File.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12250 entries, 0 to 12249
Data columns (total 36 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   NDC11                             12250 non-null  object        
 1   NDC-11 (Package)                  12250 non-null  object        
 2   NDC-9 (Product)                   12250 non-null  object        
 3   Generic Name                      12250 non-null  object        
 4   Brand Name                        12250 non-null  object        
 5   Strength                          11406 non-null  object        
 6   SEER*Rx Category                  12250 non-null  object        
 7   Major Class                       12250 non-null  object        
 8   Minor Class                       10703 non-null  object        
 9   Administration Route              11971 non-null  object        
 10  Package Effective Date            12241 non-nu

In [107]:
#SEER_NDC_File contains NDC-level information for all oncology-related medications with baseline oncology info
#Now need to add additional oncology information

In [108]:
#Can now add billable units to this file based on J Code number

In [111]:
#Download the most recent Billable Units file:
#https://www.dmepdac.com/palmetto/PDACv2.nsf/DID/0FEARUVOBV

url3 = "https://www.dmepdac.com/palmetto/PDAC_files.nsf/F/PDAC2026-04-05XWalk.xlsx/$FILE/2026-04-05XWalk.xlsx"
HCPCS_Billable = pd.read_excel(url3)

/opt/anaconda3/lib/python3.9/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [112]:
HCPCS_Billable.head()

,NDC,NDC Mod,HCPCS,HCPCS Mod,Relationship Start Date,Relationship End Date,HCPCS Description,NDC Label,Number of Items in NDC Package,NDC Package Measure,...,HCPCS Measure #1,CF,Start Date #1,End Date #1,Prior Start Date #2,Prior End Date #2,Prior Conversion\nFactor #2,Prior Start Date #3,Prior End Date #3,Prior Conversion\nFactor #3
0,00002-7335-11,NaN,J2941,NaN,03/01/2006,12/31/2020,"INJECTION, SOMATROPIN, 1 MG",HUMATROPE (WITH STERILE DILUENT) 5 MG,1.0,EA,...,MG,5.0,03/01/2006,12/31/2020,NaN,NaN,NaN,NaN,NaN,NaN
1,00002-7510-01,NaN,J1817,NaN,01/01/2003,99/99/9999,"INSULIN FOR ADMINISTRATION THROUGH DME (I.E., ...",HUMALOG (VIAL) 100 U/ML,10.0,ML,...,U,2.0,01/01/2003,99/99/9999,NaN,NaN,NaN,NaN,NaN,NaN
2,00002-7510-99,NaN,J1817,NaN,07/24/1996,99/99/9999,"INSULIN FOR ADMINISTRATION THROUGH DME (I.E., ...",HUMALOG (VIAL) 100 U/1 ML,10.0,ML,...,U,2.0,07/24/1996,99/99/9999,NaN,NaN,NaN,NaN,NaN,NaN
3,00002-7511-01,NaN,J1815,NaN,01/01/2003,99/99/9999,"INJECTION, INSULIN, PER 5 UNITS",HUMALOG MIX 75/25 (VIAL) 75 U/ML-25 U/ML,10.0,ML,...,U,20.0,01/01/2003,99/99/9999,NaN,NaN,NaN,NaN,NaN,NaN
4,00002-7511-99,NaN,J1815,NaN,12/07/2007,09/18/2025,"INJECTION, INSULIN, PER 5 UNITS",HUMALOG MIX 75/25 (VIAL) 75 U/1 ML-25 U/1 ML,10.0,ML,...,U,20.0,12/07/2007,09/18/2025,NaN,NaN,NaN,NaN,NaN,NaN


In [113]:
HCPCS_Billable = HCPCS_Billable[[
    'NDC',                             
    'HCPCS',
    'Relationship Start Date',
    'Relationship End Date',
    'HCPCS Description',
    'NDC Label',
    'Number of Items in NDC Package',
    'NDC Package Measure',
    'NDC Package Type',
    'Route of Administration',
    'Billing Units',
    'HCPCS Amount #1',
    'HCPCS Measure #1',
    'CF',
    'Start Date #1',
    'End Date #1'
]]

In [114]:
HCPCS_Billable.head()

,NDC,HCPCS,Relationship Start Date,Relationship End Date,HCPCS Description,NDC Label,Number of Items in NDC Package,NDC Package Measure,NDC Package Type,Route of Administration,Billing Units,HCPCS Amount #1,HCPCS Measure #1,CF,Start Date #1,End Date #1
0,00002-7335-11,J2941,03/01/2006,12/31/2020,"INJECTION, SOMATROPIN, 1 MG",HUMATROPE (WITH STERILE DILUENT) 5 MG,1.0,EA,VL,SC,EA,1.0,MG,5.0,03/01/2006,12/31/2020
1,00002-7510-01,J1817,01/01/2003,99/99/9999,"INSULIN FOR ADMINISTRATION THROUGH DME (I.E., ...",HUMALOG (VIAL) 100 U/ML,10.0,ML,VL,SC,ML,50.0,U,2.0,01/01/2003,99/99/9999
2,00002-7510-99,J1817,07/24/1996,99/99/9999,"INSULIN FOR ADMINISTRATION THROUGH DME (I.E., ...",HUMALOG (VIAL) 100 U/1 ML,10.0,ML,VL,IJ,ML,50.0,U,2.0,07/24/1996,99/99/9999
3,00002-7511-01,J1815,01/01/2003,99/99/9999,"INJECTION, INSULIN, PER 5 UNITS",HUMALOG MIX 75/25 (VIAL) 75 U/ML-25 U/ML,10.0,ML,VL,SC,ML,5.0,U,20.0,01/01/2003,99/99/9999
4,00002-7511-99,J1815,12/07/2007,09/18/2025,"INJECTION, INSULIN, PER 5 UNITS",HUMALOG MIX 75/25 (VIAL) 75 U/1 ML-25 U/1 ML,10.0,ML,VL,SC,ML,5.0,U,20.0,12/07/2007,09/18/2025


In [115]:
HCPCS_Billable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11244 entries, 0 to 11243
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   NDC                             11244 non-null  object 
 1   HCPCS                           11244 non-null  object 
 2   Relationship Start Date         11244 non-null  object 
 3   Relationship End Date           11244 non-null  object 
 4   HCPCS Description               11244 non-null  object 
 5   NDC Label                       11244 non-null  object 
 6   Number of Items in NDC Package  11244 non-null  float64
 7   NDC Package Measure             11244 non-null  object 
 8   NDC Package Type                8808 non-null   object 
 9   Route of Administration         10205 non-null  object 
 10  Billing Units                   11244 non-null  object 
 11  HCPCS Amount #1                 11244 non-null  float64
 12  HCPCS Measure #1                

In [116]:
#adjust date columns to datetime format

HCPCS_Billable['Relationship Start Date'] = pd.to_datetime(HCPCS_Billable['Relationship Start Date'], errors='coerce')
HCPCS_Billable['Relationship End Date'] = pd.to_datetime(HCPCS_Billable['Relationship End Date'], errors='coerce')
HCPCS_Billable['Start Date #1'] = pd.to_datetime(HCPCS_Billable['Start Date #1'], errors='coerce')
HCPCS_Billable['End Date #1'] = pd.to_datetime(HCPCS_Billable['End Date #1'], errors='coerce')

In [117]:
HCPCS_Billable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11244 entries, 0 to 11243
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   NDC                             11244 non-null  object        
 1   HCPCS                           11244 non-null  object        
 2   Relationship Start Date         11244 non-null  datetime64[ns]
 3   Relationship End Date           3168 non-null   datetime64[ns]
 4   HCPCS Description               11244 non-null  object        
 5   NDC Label                       11244 non-null  object        
 6   Number of Items in NDC Package  11244 non-null  float64       
 7   NDC Package Measure             11244 non-null  object        
 8   NDC Package Type                8808 non-null   object        
 9   Route of Administration         10205 non-null  object        
 10  Billing Units                   11244 non-null  object        
 11  HC

In [118]:
#create an NDC11 column in the HCPCS_Billable file to join on later

HCPCS_Billable['NDC11'] = np.nan 

In [119]:
#will remove the non-numberical characters from NDC-11 to create the new NDC11 column
#create the product column by removing all non-numerics (which includes "-")

HCPCS_Billable['NDC11'] = HCPCS_Billable['NDC'].str.replace(r'\D', '', regex=True)

In [120]:
#rearrange the columns

HCPCS_Billable = HCPCS_Billable[[
    'NDC11',
    'HCPCS',
    'Relationship Start Date',
    'Relationship End Date',
    'HCPCS Description',
    'NDC Label',
    'Number of Items in NDC Package',
    'NDC Package Measure',
    'NDC Package Type',
    'Route of Administration',
    'Billing Units',
    'HCPCS Amount #1',
    'HCPCS Measure #1',
    'CF',
    'Start Date #1',
    'End Date #1'
]]

In [121]:
#merge HCPCS_Billable file into SEER_NDC_File to acquire billing information for all J-Code related meds

In [122]:
SEER_Billable = pd.merge(SEER_NDC_File, HCPCS_Billable, on='NDC11', how='left')

In [123]:
SEER_Billable.head()

,NDC11,NDC-11 (Package),NDC-9 (Product),Generic Name,Brand Name,Strength,SEER*Rx Category,Major Class,Minor Class,Administration Route,...,Number of Items in NDC Package,NDC Package Measure,NDC Package Type,Route of Administration,Billing Units,HCPCS Amount #1,HCPCS Measure #1,CF,Start Date #1,End Date #1
0,00002171728,00002-1717-28,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
1,00002171756,00002-1717-56,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
2,00002171761,00002-1717-61,00002-1717,Imlunestrant,Inluriyo,200.0 mg/1,Hormonal Therapy,Estrogen Receptor Antagonist,ER⍺,Oral,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
3,00002298026,00002-2980-26,00002-2980,Selpercatinib,RETEVMO,80.0 mg/1,Chemotherapy,Tyrosine Kinase Inhibitor,VEGFR,Oral,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
4,00002298060,00002-2980-60,00002-2980,Selpercatinib,RETEVMO,80.0 mg/1,Chemotherapy,Tyrosine Kinase Inhibitor,VEGFR,Oral,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT


In [124]:
SEER_Billable.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12401 entries, 0 to 12400
Data columns (total 51 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   NDC11                             12401 non-null  object        
 1   NDC-11 (Package)                  12401 non-null  object        
 2   NDC-9 (Product)                   12401 non-null  object        
 3   Generic Name                      12401 non-null  object        
 4   Brand Name                        12401 non-null  object        
 5   Strength                          11554 non-null  object        
 6   SEER*Rx Category                  12401 non-null  object        
 7   Major Class                       12401 non-null  object        
 8   Minor Class                       10852 non-null  object        
 9   Administration Route              12121 non-null  object        
 10  Package Effective Date            12392 non-nu

In [125]:
#Rearrange columns to download

SEER_Billable = SEER_Billable[['Brand Name',
                                       'Generic Name',
                                       'Strength',
                                       'ACTIVE_NUMERATOR_STRENGTH',
                                       'ACTIVE_INGRED_UNIT', 
                                       'DOSAGEFORMNAME', 
                                       'ROUTENAME',
                                       'Route of Administration',
                                       'Administration Route',
                                       'PHARM_CLASSES',
                                       'SEER*Rx Category',
                                       'Major Class',
                                       'Minor Class',
                                       'PROPRIETARYNAME',
                                       'PROPRIETARYNAMESUFFIX',
                                       'NONPROPRIETARYNAME',
                                       'SUBSTANCENAME',
                                       'DEASCHEDULE',
                                       'MARKETINGCATEGORYNAME',
                                       'Status',
                                       'NDC11',
                                       'NDC-11 (Package)',
                                       'NDC-9 (Product)',
                                       'NDCProduct',
                                       'PACKAGEDESCRIPTION',
                                       'PRODUCTTYPENAME',
                                       'APPLICATIONNUMBER',
                                       'LABELERNAME',
                                       'Package Effective Date',
                                       'Package Discontinuation Date',
                                       'Relationship Start Date',
                                       'Relationship End Date',
                                       'LISTING_RECORD_CERTIFIED_THROUGH',
                                       'NDC Label',
                                       'Number of Items in NDC Package',
                                       'NDC Package Measure',
                                       'NDC Package Type',
                                       'Billing Units',
                                       'HCPCS Amount #1',
                                       'HCPCS Measure #1',
                                       'Start Date #1',
                                       'End Date #1',
                                      ]]

In [126]:
SEER_Billable.head()

,Brand Name,Generic Name,Strength,ACTIVE_NUMERATOR_STRENGTH,ACTIVE_INGRED_UNIT,DOSAGEFORMNAME,ROUTENAME,Route of Administration,Administration Route,PHARM_CLASSES,...,LISTING_RECORD_CERTIFIED_THROUGH,NDC Label,Number of Items in NDC Package,NDC Package Measure,NDC Package Type,Billing Units,HCPCS Amount #1,HCPCS Measure #1,Start Date #1,End Date #1
0,Inluriyo,Imlunestrant,200.0 mg/1,200,mg/1,"TABLET, FILM COATED",ORAL,NaN,Oral,Breast Cancer Resistance Protein Inhibitors [M...,...,20271231.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
1,Inluriyo,Imlunestrant,200.0 mg/1,200,mg/1,"TABLET, FILM COATED",ORAL,NaN,Oral,Breast Cancer Resistance Protein Inhibitors [M...,...,20271231.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
2,Inluriyo,Imlunestrant,200.0 mg/1,200,mg/1,"TABLET, FILM COATED",ORAL,NaN,Oral,Breast Cancer Resistance Protein Inhibitors [M...,...,20271231.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
3,RETEVMO,Selpercatinib,80.0 mg/1,80,mg/1,CAPSULE,ORAL,NaN,Oral,Breast Cancer Resistance Protein Inhibitors [M...,...,20261231.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
4,RETEVMO,Selpercatinib,80.0 mg/1,80,mg/1,CAPSULE,ORAL,NaN,Oral,Breast Cancer Resistance Protein Inhibitors [M...,...,20261231.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT


In [127]:
SEER_Billable.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12401 entries, 0 to 12400
Data columns (total 42 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Brand Name                        12401 non-null  object        
 1   Generic Name                      12401 non-null  object        
 2   Strength                          11554 non-null  object        
 3   ACTIVE_NUMERATOR_STRENGTH         5598 non-null   object        
 4   ACTIVE_INGRED_UNIT                5598 non-null   object        
 5   DOSAGEFORMNAME                    5715 non-null   object        
 6   ROUTENAME                         5630 non-null   object        
 7   Route of Administration           2514 non-null   object        
 8   Administration Route              12121 non-null  object        
 9   PHARM_CLASSES                     5364 non-null   object        
 10  SEER*Rx Category                  12401 non-nu

In [128]:
#Download the Master Drug File to inspect/use 

#SEER_Billable.to_csv (r'/Your_File_Path/Master_Drug_File.csv', index=None)